# Eval Baseline PaddleOCR
Dataset: Teklia/IAM-line — test[:1000]
Metrik: CER & WER (apple-to-apple dengan DeepSeek OCR2)

## Installation
> **Kaggle T4**: gunakan paddlepaddle-gpu agar inferensi cepat.

In [ ]:
%%capture
# PaddlePaddle GPU build untuk CUDA 11.x (default Kaggle T4)
!pip install paddlepaddle-gpu==2.6.2.post120 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
!pip install paddleocr>=2.9.1
!pip install "datasets==4.3.0"
!pip install jiwer
!pip install pillow numpy


## Import

In [ ]:
import re
import os
import json
import numpy as np
from PIL import Image
from datasets import load_dataset
from paddleocr import PaddleOCR
from jiwer import cer, wer


## Load Dataset

In [ ]:
print("Loading dataset...")
baseline_dataset = load_dataset("Teklia/IAM-line", split="test[:1000]")

# Preview
sample = baseline_dataset[0]
sample["image"].save("temp_paddle.jpg")
print("GT contoh pertama:", sample["text"])
sample["image"]


## Normalisasi Teks
> Sama persis dengan DeepSeek OCR2 baseline

In [ ]:
def normalize_text(text):
    text = text.strip()
    text = text.replace("
", " ")       # newline → space
    text = re.sub(r" +", " ", text)      # multi-space → single space
    text = text.lower()                  # lowercase (case-insensitive)
    return text


## Load Model PaddleOCR
>  — IAM-line sudah lurus, tidak perlu deteksi rotasi.
>  — memanfaatkan T4 GPU di Kaggle.

In [ ]:
import paddle
USE_GPU = paddle.device.is_compiled_with_cuda()  # True otomatis di Kaggle T4
print(f"GPU available: {USE_GPU}")

ocr_engine = PaddleOCR(
    use_angle_cls=False,  # IAM-line: baris lurus, tidak perlu angle cls
    lang="en",
    use_gpu=USE_GPU,
    show_log=False,
)
print("PaddleOCR loaded.")


## Evaluasi PaddleOCR Baseline
> Loop identik dengan DeepSeek OCR2: per-sample CER/WER, akumulasi, lalu rata-rata

In [ ]:
baseline_total_cer = 0
baseline_total_wer = 0

for i, sample in enumerate(baseline_dataset):
    image = sample["image"]
    if image.mode != "RGB":
        image = image.convert("RGB")

    gt = normalize_text(sample["text"])

    # ---- Inferensi PaddleOCR ----
    img_array = np.array(image)
    result = ocr_engine.ocr(img_array, cls=False)

    # Kumpulkan semua token teks yang terdeteksi pada baris gambar
    lines = []
    if result and result[0]:
        for line in result[0]:
            lines.append(line[1][0])  # (text, confidence) → ambil text
    pred_raw = " ".join(lines)
    pred = normalize_text(pred_raw)

    # Guard: hindari empty string yang membuat jiwer error
    if len(gt.strip()) == 0:
        gt = " "
    if len(pred.strip()) == 0:
        pred = " "

    # ---- Hitung Metrik (identik DeepSeek) ----
    baseline_total_cer += cer(gt, pred)
    baseline_total_wer += wer(gt, pred)

    if (i + 1) % 50 == 0 or i == 0:
        print(f"[{i+1}/{len(baseline_dataset)}]")
        print(f"  GT  : {gt}")
        print(f"  Pred: {pred}")
        print(f"  CER : {cer(gt, pred):.4f} | WER : {wer(gt, pred):.4f}
")

print(f"Done {i+1}/{len(baseline_dataset)}

")


## Hasil & Simpan

In [ ]:
# Hitung rata-rata  (sama persis dengan DeepSeek OCR2)
baseline_avg_cer = baseline_total_cer / len(baseline_dataset)
baseline_avg_wer = baseline_total_wer / len(baseline_dataset)

print("
=== PADDLEOCR BASELINE RESULT ===")
print("Baseline CER:", baseline_avg_cer)
print("Baseline WER:", baseline_avg_wer)

# ---- Simpan TXT ----
output_dir = "/kaggle/working"
os.makedirs(output_dir, exist_ok=True)

with open(f"{output_dir}/paddleocr_baseline_results.txt", "w") as f:
    f.write("=== PADDLEOCR BASELINE RESULT ===
")
    f.write(f"Model       : PaddleOCR (lang=en, use_angle_cls=False)
")
    f.write(f"Baseline CER: {baseline_avg_cer}
")
    f.write(f"Baseline WER: {baseline_avg_wer}
")
print(f"Hasil disimpan ke {output_dir}/paddleocr_baseline_results.txt")

# ---- Simpan JSON ----
baseline_results = {
    "model": "PaddleOCR",
    "config": {
        "lang": "en",
        "use_angle_cls": False,
        "use_gpu": USE_GPU,
    },
    "test_split": "test[:1000]",
    "num_samples": len(baseline_dataset),
    "avg_cer": baseline_avg_cer,
    "avg_wer": baseline_avg_wer,
}
with open(f"{output_dir}/paddleocr_baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=4)
print(f"Hasil disimpan ke {output_dir}/paddleocr_baseline_results.json")
